# Qui uso il Firms NASA per fare la previsione di incendi sul territorio

## API per vedere la mia MAP_KEY per accedere ai dati via richieste API

In [ ]:
# # Let's set your map key that was emailed to you. It should look something like 'abcdef1234567890abcdef1234567890'
# MAP_KEY = '<replace with your map_key>'
MAP_KEY = 'd2678645bce596b35dcb6e57a24a82f4'

# now let's check how many transactions we have
import pandas as pd
import requests
url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
try:
  response = requests.get(url)
  data = response.json()
  df = pd.Series(data)
  display(df)
except:
  # possible error, wrong MAP_KEY value, check for extra quotes, missing letters
  print ("There is an issue with the query. \nTry in your browser: %s" % url)

## API per vedere i datasets

In [ ]:
# let's query data_availability to find out what date range is available for various datasets
# we will explain these datasets a bit later

# this url will return information about all supported sensors and their corresponding datasets
# instead of 'all' you can specify individual sensor, ex:LANDSAT_NRT
da_url = f"https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{MAP_KEY}/all"
df_datasets = pd.read_csv(da_url)
display(df_datasets)


## Script per salvare tutti i dataset per iterare sopra

In [ ]:
datasets = []
for i in df_datasets["data_id"]:
    datasets.append(i)
print(datasets)

In [ ]:
print(df_datasets.columns.tolist())
display(df_datasets.head())

## Data Ingest 

In [ ]:
from datetime import date
from pathlib import Path
import sys

# Eseguire dal folder matteo oppure dalla root del repository.
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "cambiamenti_climatici/ingestion/firms.py").exists())
sys.path.insert(0, str(root / "cambiamenti_climatici/ingestion"))
from firms import acquire

# Fonte esplicita: non concatenare automaticamente NRT/SP o sensori diversi.
source = "MODIS_SP"
# Intervallo storico del progetto; modificarlo esplicitamente se necessario.
start, end = date(2025, 4, 26), date(2026, 7, 17)
output = root / "cambiamenti_climatici/ingestion/output/modis_sp_history"
coverage = acquire(MAP_KEY, source, start, end, output)
unresolved = [r for r in coverage if r["status"] in {"failed", "unavailable"}]
print(f"Giorni richiesti: {len(coverage)}; non risolti: {len(unresolved)}")
# Le date fuori disponibilità restano esplicite in coverage.csv.
if unresolved:
    raise RuntimeError("Copertura incompleta: consultare coverage.csv prima di esportare.")
df_finale = pd.read_csv(output / "recovered.csv")
print("Record recuperati:", len(df_finale))


In [ ]:
df_finale.shape

In [ ]:
df = df_finale.to_csv("dataset_incendi_FIRMS.csv", index=False)
df = df_finale